# ZeroTrust-AI — Kaggle Training Notebook
## Hierarchical Meta-Fuser: Transformer + Autoencoder + GAN + MLP

| Phase | Task | Estimated Time |
|-------|------|----------------|
| 1 | Transformer Lens (20 epochs, 240k samples) | ~15-20 min |
| 2 | 3-Lens Score Map Generation | ~3-5 min |
| 3 | MLP Meta-Fuser Training | ~2-3 min |
| **TOTAL** | | **~20-28 min** |

> **Before running:** Enable **GPU T4 x2** in the Settings panel on the right side of Kaggle.

## Cell 1 — Install Dependencies

In [ ]:
# Uncomment if running on a fresh Kaggle environment
# !pip install scikit-learn joblib pandas numpy torch -q
print("Dependencies ready.")

## Cell 2 — Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import RobustScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
import joblib, warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('GPUs:', torch.cuda.device_count())

## Cell 3 — Load and Prepare Dataset

> Upload `final_balanced_240k_fixed.csv` to Kaggle as a dataset,
> then update `DATASET_PATH` below to match your dataset name.

In [ ]:
# UPDATE THIS PATH after uploading your dataset to Kaggle
DATASET_PATH = '/kaggle/input/zerotrust-dataset/final_balanced_240k_fixed.csv'

print('Loading dataset...')
df = pd.read_csv(DATASET_PATH)
print('Shape:', df.shape)
print(df['label'].value_counts())

len_cols = [f'splt_len_{i}' for i in range(1, 21)]
iat_cols = [f'splt_iat_{i}' for i in range(1, 21)]

len_features = df[len_cols].values.astype('float32')
iat_features = df[iat_cols].values.astype('float32')
labels       = df['label'].values.astype('float32')

scaler_len = RobustScaler()
scaler_iat = RobustScaler()
len_features = scaler_len.fit_transform(len_features)
iat_features = scaler_iat.fit_transform(iat_features)
joblib.dump(scaler_len, 'scaler_len.joblib')
joblib.dump(scaler_iat, 'scaler_iat.joblib')
print('Scalers saved.')

# Stack into [N, 20, 2]
X_sequences = np.stack([len_features, iat_features], axis=-1)
print('Sequence shape:', X_sequences.shape)

X_tensor = torch.FloatTensor(X_sequences)
y_tensor = torch.FloatTensor(labels).view(-1, 1)

dataset    = TensorDataset(X_tensor, y_tensor)
train_size = int(0.85 * len(dataset))
val_size   = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {train_size}  |  Val: {val_size}')

## Cell 4 — 1D-Transformer Lens (Model Definition)

Multi-Head Self-Attention over 20 packet events.  
Each token = one packet (length + inter-arrival time).

In [ ]:
class NetworkTransformerLens(nn.Module):
    def __init__(self, feature_dim=2, embedding_dim=64, nhead=4, num_layers=3):
        super().__init__()
        self.input_projection = nn.Linear(feature_dim, embedding_dim)
        self.pos_encoder       = nn.Parameter(torch.randn(1, 20, embedding_dim))
        self.dropout           = nn.Dropout(0.1)
        enc = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=nhead,
            dim_feedforward=128, dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers)
        self.classifier  = nn.Sequential(
            nn.Linear(embedding_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),  nn.Sigmoid()
        )

    def forward(self, x):
        x = self.input_projection(x) + self.pos_encoder
        x = self.dropout(x)
        x = self.transformer(x)
        x = x.mean(dim=1)          # global average pool
        return self.classifier(x)

m = NetworkTransformerLens()
print('Model params:', sum(p.numel() for p in m.parameters()))

## Cell 5 — Train Transformer Lens

> **Estimated time: ~15-20 min on T4 x2**

In [ ]:
def train_transformer(model, train_loader, val_loader, epochs=20, lr=3e-4):
    model     = nn.DataParallel(model).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.BCELoss()
    best_auc  = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        tloss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tloss += loss.item()

        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                p = model(Xb.to(DEVICE)).cpu().numpy()
                preds.extend(p.flatten())
                labs.extend(yb.numpy().flatten())

        auc = roc_auc_score(labs, preds)
        acc = accuracy_score(labs, (np.array(preds) > 0.5).astype(int))
        scheduler.step()
        print(f'Epoch {epoch:02d}/{epochs}  loss={tloss/len(train_loader):.4f}  AUC={auc:.4f}  Acc={acc:.4f}')

        if auc > best_auc:
            best_auc = auc
            torch.save(model.module.state_dict(), 'network_transformer_lens.pth')
            print(f'  -- best saved (AUC={best_auc:.4f})')

    print('Done! Best AUC:', round(best_auc, 4))
    return model

transformer_model = NetworkTransformerLens()
transformer_model = train_transformer(transformer_model, train_loader, val_loader, epochs=20)

## Cell 6 — Load Pretrained Autoencoder

> Upload `autoencoder.pth` to Kaggle as a dataset, then update the path below.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim=40):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 8))
        self.decoder = nn.Sequential(
            nn.Linear(8, 16), nn.ReLU(),
            nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, input_dim))
    def forward(self, x): return self.decoder(self.encoder(x))
    def get_score(self, x): return torch.mean((x - self.forward(x))**2, dim=1, keepdim=True)

AE_PATH  = '/kaggle/input/zerotrust-models/autoencoder.pth'  # UPDATE THIS
ae_model = Autoencoder(40).to(DEVICE)
ae_model.load_state_dict(torch.load(AE_PATH, map_location=DEVICE))
ae_model.eval()
print('Autoencoder loaded.')

X_flat        = np.concatenate([len_features, iat_features], axis=1)
X_flat_tensor = torch.FloatTensor(X_flat)

## Cell 7 — Load Pretrained GAN Discriminator

> Upload `gan_discriminator_lens.pth` to Kaggle as a dataset, then update the path below.

In [ ]:
class GANDiscriminator(nn.Module):
    def __init__(self, input_dim=40):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(32, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x)

GAN_PATH  = '/kaggle/input/zerotrust-models/gan_discriminator_lens.pth'  # UPDATE THIS
gan_model = GANDiscriminator(40).to(DEVICE)
gan_model.load_state_dict(torch.load(GAN_PATH, map_location=DEVICE))
gan_model.eval()
print('GAN Discriminator loaded.')

## Cell 8 — Generate 3-Lens Score Map

> **Estimated time: ~3-5 min**

In [ ]:
def get_scores(model, tensor, device, batch=1024):
    scores, loader = [], DataLoader(TensorDataset(tensor), batch_size=batch, shuffle=False, num_workers=2)
    with torch.no_grad():
        for (Xb,) in loader:
            scores.extend(model(Xb.to(device)).cpu().numpy().flatten())
    return np.array(scores)

best_tf = NetworkTransformerLens().to(DEVICE)
best_tf.load_state_dict(torch.load('network_transformer_lens.pth', map_location=DEVICE))
best_tf.eval()

print('Scoring all 3 lenses over full dataset...')
transformer_scores = get_scores(best_tf, X_tensor, DEVICE)
ae_scores          = get_scores(ae_model, X_flat_tensor, DEVICE)
gan_scores         = get_scores(gan_model, X_flat_tensor, DEVICE)
print('Scores generated.')

fusion_df = pd.DataFrame({
    'transformer_score': transformer_scores,
    'ae_score':          ae_scores,
    'gan_score':         gan_scores,
    'label':             labels
})
fusion_df.to_csv('kaggle_fusion.csv', index=False)
print('Fusion CSV saved. Shape:', fusion_df.shape)
fusion_df.describe()

## Cell 9 — Train Stage-2 MLP Meta-Fuser

> **Estimated time: ~2-3 min**  |  Adaptive threshold: **0.40**

In [ ]:
X_fusion = fusion_df[['transformer_score', 'ae_score', 'gan_score']].values
y_fusion = fusion_df['label'].values
X_tr, X_te, y_tr, y_te = train_test_split(
    X_fusion, y_fusion, test_size=0.15, random_state=42, stratify=y_fusion)

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32, 16), activation='relu', solver='adam',
    alpha=1e-4, learning_rate_init=1e-3, max_iter=300,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=15,
    random_state=42, verbose=True)
mlp.fit(X_tr, y_tr)
joblib.dump(mlp, 'hierarchical_meta_fuser_gan.joblib')
print('Saved: hierarchical_meta_fuser_gan.joblib')

## Cell 10 — Final Evaluation

In [ ]:
y_prob = mlp.predict_proba(X_te)[:, 1]
y_pred = (y_prob > 0.40).astype(int)

print('=' * 60)
print('FINAL HIERARCHICAL META-FUSER RESULTS')
print('=' * 60)
print(classification_report(y_te, y_pred, target_names=['Benign', 'Malicious']))
print('ROC-AUC :', round(roc_auc_score(y_te, y_prob), 4))
print('Accuracy:', round(accuracy_score(y_te, y_pred), 4))
cm = confusion_matrix(y_te, y_pred)
print('Confusion Matrix:\n', cm)

## Cell 11 — Download Output Files

After training finishes, go to the **Output tab** on the right and download:

| File | Deploy To |
|------|-----------|
| `network_transformer_lens.pth` | `c2_ddos/scripts/models/` |
| `hierarchical_meta_fuser_gan.joblib` | `c2_ddos/scripts/models/` |
| `scaler_len.joblib` | `c2_ddos/scripts/models/` |
| `scaler_iat.joblib` | `c2_ddos/scripts/models/` |

In [ ]:
files = [
    ('network_transformer_lens.pth',       'Transformer weights'),
    ('hierarchical_meta_fuser_gan.joblib', 'MLP Meta-Fuser'),
    ('scaler_len.joblib',                  'Length scaler'),
    ('scaler_iat.joblib',                  'IAT scaler'),
    ('kaggle_fusion.csv',                  'Fusion score map'),
]
print('Output files:')
for f, desc in files:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'  [{status}]  {f:<45}  {desc}')